# Step 6 — Evaluating a Model's Generated Sample

After fine-tuning, you sample new molecules from the model and ask two questions:

1. **Did the model learn the fine-tuning distribution?**
   Compare the generated set against the fine-tuning reference (train + val).
2. **Does the model generalise to unseen chemistry?**
   Compare the generated set against the held-out test set — molecules that
   were never touched during training or augmentation.

| Reference | Role |
|---|---|
| **Fine-tuning set** | Canonical train + val molecules — the distribution the model was trained toward |
| **Test set** | Held-out canonical molecules — an honest measure of generalisation |
| **Generated sample** | Molecules sampled from the fine-tuned model |

> **Note on canonical SMILES and set sizes:** Always use canonical,
> deduplicated SMILES for the reference sets — not the augmented versions
> used during training.  Report n as the number of original molecules, not
> the number of augmented representations.

**What you will do:**
1. Load the three molecule sets
2. Evaluate basic quality of the generated sample (validity, uniqueness)
3. Measure novelty relative to each reference
4. Compare property distributions (generated vs each reference)
5. Measure structural distances (generated vs each reference)
6. Summarise all metrics in a single comparison table

**Dummy data is provided** so you can run the notebook end-to-end before
bringing your own model output.  Replace each list with your actual data.

For the mathematical definition and interpretation of every metric used in this notebook, see [`docs/metrics_reference.md`](docs/metrics_reference.md).

## Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from evaluation import load_smiles
from evaluation.metrics import (
    validity,
    uniqueness,
    novelty,
    mean_pairwise_distance,
    scaffold_entropy,
    nearest_neighbor_distance,
    fraction_passing_lipinski,
)
from evaluation.properties import compute_properties
from evaluation.visualization import (
    draw_molecule_grid,
    plot_distribution_comparison,
    compare_distributions,
)

print("Imports OK")

In [ ]:
# Automatically detect the directory where this notebook is located
base_dir = os.path.dirname(os.path.abspath("__file__"))

---
## Load the Three Sets

**What to load:**
- `FINETUNING_SMILES` — canonical SMILES of the molecules you fine-tuned on
  (train + val combined, deduplicated, *not* the augmented versions)
- `TEST_SMILES` — the held-out canonical test set (never used in training or augmentation)
- `GENERATED_SMILES` — raw output from your fine-tuned model; keep duplicates
  and any invalid SMILES — that information is part of the evaluation

In [ ]:
# ── Fine-tuning reference set ─────────────────────────────────────────────────
# Load the canonical train + val molecules (deduplicated). Not the augmented versions!

data_path = os.path.join(base_dir, "your_path_here")

FINETUNING_SMILES = load_smiles(f"{data_path}/train.smi") + load_smiles(f"{data_path}/val.smi")

print(f"Fine-tuning reference : {len(FINETUNING_SMILES)} molecules (canonical, deduplicated)")

In [ ]:
# ── Held-out test set ─────────────────────────────────────────────────────────
# Canonical SMILES, never seen during training or augmentation.
TEST_SMILES = load_smiles(f"{data_path}/test.smi")

# Note: report statistics using n = len(TEST_SMILES), the number of original
# molecules — not the number of augmented representations.

print(f"Test set (held-out)   : {len(TEST_SMILES)} molecules (canonical, n = original count)")

In [ ]:
# ── Generated sample ──────────────────────────────────────────────────────────
# Molecules sampled from your fine-tuned model.
sampled_path = os.path.join(base_dir, "your_path_here")

sampled_path = os.path.join(base_dir, "results/finetuning/unclean_sampled_molecules.json")
GENERATED_SMILES = load_smiles(sampled_path)

print(f"Generated sample      : {len(GENERATED_SMILES)} SMILES strings")

---
## Step 1 — Basic Quality of the Generated Sample

Before any deeper analysis, check that the model is producing valid and
unique molecules.

In [ ]:
v = validity(GENERATED_SMILES)
u = uniqueness(GENERATED_SMILES)

print(f"Validity   : {v:.3f}  ({int(v * len(GENERATED_SMILES))}/{len(GENERATED_SMILES)} valid)")
print(f"Uniqueness : {u:.3f}  (computed over valid molecules)")
print()
if v < 0.9:
    print("Warning: validity is below 0.9. The model is producing many invalid SMILES.")
if u < 0.8:
    print("Warning: uniqueness is below 0.8. The model is generating many repeated molecules.")

In [ ]:
# ── Draw a grid of the valid generated molecules ──────────────────────────────
image = draw_molecule_grid(GENERATED_SMILES, n_cols=5)
display(image)

---
## Step 2 — Novelty

Novelty measures what fraction of generated molecules are not exact canonical
matches to a reference set.  We compute it against both references:

- **vs. fine-tuning set** — did the model simply memorise its training data?
  Some reproduction is expected and healthy; full memorisation (novelty = 0)
  means the model adds no generative value.
- **vs. test set** — are the generated molecules genuinely new relative to
  the held-out set?  A high value here suggests the model is sampling from a
  broader space than the specific test molecules.

In [ ]:
novelty_vs_finetune = novelty(GENERATED_SMILES, reference=FINETUNING_SMILES)
novelty_vs_test     = novelty(GENERATED_SMILES, reference=TEST_SMILES)

print(f"Novelty vs. fine-tuning set : {novelty_vs_finetune:.3f}")
print(f"Novelty vs. test set        : {novelty_vs_test:.3f}")
print()
print(f"  {(1 - novelty_vs_finetune)*100:.1f}% of valid generated molecules are exact matches to the fine-tuning set.")
print(f"  {(1 - novelty_vs_test)*100:.1f}% of valid generated molecules are exact matches to the test set.")

---
## Step 3 — Property Distributions

A well-tuned model should shift its property distribution towards the
fine-tuning data while staying within drug-like ranges.  Compare the
generated set to both references.

In [ ]:
props_generated  = compute_properties(GENERATED_SMILES)
props_finetuning = compute_properties(FINETUNING_SMILES)
props_test       = compute_properties(TEST_SMILES)

print("Generated sample — property summary:")
props_generated.describe().round(2)

In [ ]:
# ── Generated vs. fine-tuning reference ──────────────────────────────────────
n_props = len(props_generated.columns)
n_cols = min(4, n_props)
n_rows = (n_props + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(props_generated.columns):
    plot_distribution_comparison(
        props_finetuning[col], props_generated[col],
        labels=["Fine-tuning", "Generated"],
        xlabel=col.replace("_", " ").title(),
        ax=axes[i]
    )
plt.suptitle("Generated vs. Fine-tuning Reference", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Generated vs. test set (held-out) ─────────────────────────────────────────
n_props = len(props_generated.columns)
n_cols = min(4, n_props)
n_rows = (n_props + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(props_generated.columns):
    plot_distribution_comparison(
        props_test[col], props_generated[col],
        labels=["Test set", "Generated"],
        xlabel=col.replace("_", " ").title(),
        ax=axes[i]
    )
plt.suptitle("Generated vs. Held-out Test Set", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 4 — Structural Distance

Property distributions describe *physicochemical* similarity.  Nearest-neighbour
distance (NND) measures *fingerprint-level* structural similarity — how closely
generated molecules resemble each reference set at the level of atomic environments.

We also compute the mean pairwise distance within each set as a measure of
internal diversity.  A well-tuned model should produce a generated set with
diversity comparable to the fine-tuning data.

In [ ]:
generated_mpd  = mean_pairwise_distance(GENERATED_SMILES)
finetuning_mpd = mean_pairwise_distance(FINETUNING_SMILES)
test_mpd       = mean_pairwise_distance(TEST_SMILES)

print("Mean pairwise distance (internal structural diversity):")
print(f"  Generated sample : {generated_mpd:.3f}")
print(f"  Fine-tuning set  : {finetuning_mpd:.3f}")
print(f"  Test set         : {test_mpd:.3f}")

Fine-tuning on a focused chemical series typically reduces diversity relative to the pre-trained model — a small MPD for the generated set is expected and appropriate if the fine-tuning set itself is focused.

In [ ]:
nnd_to_finetune = nearest_neighbor_distance(GENERATED_SMILES, reference=FINETUNING_SMILES)
nnd_to_test     = nearest_neighbor_distance(GENERATED_SMILES, reference=TEST_SMILES)

print("Nearest-neighbour distance (generated → reference):")
print(f"  Generated → fine-tuning set : {nnd_to_finetune:.3f}")
print(f"  Generated → test set        : {nnd_to_test:.3f}")

Interpretation: a lower NND to the fine-tuning set than to the test set indicates that fine-tuning successfully shifted the model toward the target chemical space, rather than the broader pre-training distribution.

---
## Step 5 — Statistical Comparison

Formally test whether the generated property distributions match the
fine-tuning reference (the target).

In [ ]:
# ── KS tests: generated vs. fine-tuning reference ─────────────────────────────
rows = []
for prop in props_generated.columns:
    result = compare_distributions(
        props_finetuning[prop], props_generated[prop], test="ks"
    )
    rows.append({
        "property"   : prop,
        "statistic"  : round(result["statistic"], 3),
        "p_value"    : round(result["p_value"], 3),
        "significant": result["significant"],
    })

df_tests = pd.DataFrame(rows)
print("KS test: generated vs. fine-tuning reference")
print("(p < 0.05 = distributions are significantly different)")
df_tests

---
## Step 6 — Summary Table

A single table summarising all metrics for quick comparison.

In [ ]:
lipinski_ft   = fraction_passing_lipinski(FINETUNING_SMILES)
lipinski_test = fraction_passing_lipinski(TEST_SMILES)
lipinski_gen  = fraction_passing_lipinski(GENERATED_SMILES)
se_gen        = scaffold_entropy(GENERATED_SMILES)
se_ft         = scaffold_entropy(FINETUNING_SMILES)

summary = pd.DataFrame({
    "Metric": [
        "Validity",
        "Uniqueness",
        "Novelty vs. fine-tuning set",
        "Novelty vs. test set",
        "Mean pairwise distance",
        "Scaffold entropy",
        "NND → fine-tuning set",
        "NND → test set",
        "Fraction passing Lipinski",
    ],
    "Generated": [
        round(v, 3),
        round(u, 3),
        round(novelty_vs_finetune, 3),
        round(novelty_vs_test, 3),
        round(generated_mpd, 3),
        round(se_gen, 3),
        round(nnd_to_finetune, 3),
        round(nnd_to_test, 3),
        round(lipinski_gen, 3),
    ],
    "Fine-tuning ref.": [
        "—", "—", "—", "—",
        round(finetuning_mpd, 3),
        round(se_ft, 3),
        "—", "—",
        round(lipinski_ft, 3),
    ],
    "Test set": [
        "—", "—", "—", "—",
        round(test_mpd, 3),
        "—",
        "—", "—",
        round(lipinski_test, 3),
    ],
})
summary

---
## Student Challenges

---

### Challenge 1 — Interpret the summary table

Look at the summary table above using your own model output.

- **Validity**: is it above 0.9?  A low validity suggests the model has
  not converged well, or the temperature used for sampling is too high.
- **Novelty vs. fine-tuning set**: some memorisation is expected — the model
  *should* have learned these molecules.  Complete memorisation (≈ 0) is a
  sign of overfitting.  What fraction do you consider acceptable?
- **NND to fine-tuning vs. NND to test set**: which is smaller?  If NND to
  fine-tuning < NND to test set, fine-tuning successfully biased the model
  toward the target space.

### Challenge 2 — Before vs. after fine-tuning

Sample from both the **pre-trained** model (no fine-tuning) and your
**fine-tuned** model.  For each sample compute the full metrics table.

- Did fine-tuning shift NND to the fine-tuning set downward?
- Did validity change?  (Fine-tuning on a focused set sometimes reduces
  validity if the model over-specialises.)
- Did scaffold entropy decrease?  A focused fine-tuning set should produce
  a more concentrated scaffold distribution.
- Which temperature produced the best balance between validity and novelty?

In [ ]:
# Load output from your pre-trained model (before fine-tuning) and re-run all
# metrics to compare against your fine-tuned output above.
# PT_GENERATED = load_smiles("pt_sample.smi")